# MLFlow Lab Notebook

In [6]:
! python -c "import mlflow, sklearn, pandas; print(f'mlflow: {mlflow.__version__}\n sklearn: {sklearn.__version__}\n pandas: {pandas.__version__}' )"


mlflow: 3.12.0
 sklearn: 1.8.0
 pandas: 2.3.3


3.2 - I use uv for package management, thus I don't need requirements.txt

question 1:  
we separate raw and processed data so we can explore with data without losing the original one (reproducibility, error recovery, different processing paths).  
if we modify the raw data directly, we lose informations about it, in a way that is irreversible, we can never go back to the original state of data. 

question 2 : 
because 0 in these cases represent missing values incoding and not a real possible value for the records. 
keeping it will make the model think that it's a possible value and affect its predictions (affect distance calculations for distance based models, and splitting decisions for trees.)

In [ ]:
# first training run
! uv run ../src/train.py

Chargement du dataset Pima Indians Diabetes...
Dataset charge : 768 lignes x 9 colonnes
Distribution : {0: 500, 1: 268}
Train : 614 | Test : 154
Entrainement : RF_baseline
Resultats : {'accuracy': 0.7597, 'precision': 0.6809, 'recall': 0.5926, 'f1_score': 0.6337, 'roc_auc': 0.8147}
Modele sauvegarde : /home/zeco/Desktop/TPs/mlops/diabete-mlops/models/RF_baseline.pkl


question 3 :  
pipelines are clean and self documenting, so it's easy to maintain them. they also help avoiding data leakage, and assure that the transformations applied during training will be applied during inference. 

In [12]:
# run training with MLFlow
! uv run ../src/train.py 

Chargement du dataset Pima Indians Diabetes...
Dataset charge : 768 lignes x 9 colonnes
Distribution : {0: 500, 1: 268}
Train : 614 | Test : 154
Entrainement : RF_baseline
/home/zeco/Desktop/TPs/mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/07 18:52:56 WARNING mlflow.sklearn: Saving s

question 4.a:  
metrics over the training set tell us whether the model has learned enough or not (minimize the bias)  
metrics over the testing set tell us whether the model is generalizing or not (minimize variance)
  
comparing metrics from training and testing sets enable us to detect, underfitting, overfitting, and data leakage. 

question 4.b:  
a model's signature is a way to incode the expected schema of the model: the expected input, expected output, and also possible optional params (like temperature for LLMs)   
the model's signature solves the problem of schema mismatch between training and inference, and enables better validation and errors. 

In [23]:
# multiple experiment runs for comparizon using MLFlow
! uv run ../run_experiments.py

2026/05/07 20:55:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/07 20:55:10 INFO mlflow.store.db.utils: Updating database tables
2026/05/07 20:55:15 INFO mlflow.tracking.fluent: Experiment with name 'diabetes_classification' does not exist. Creating a new experiment.
LANCEMENT DES EXPERIENCES MLflow
Chargement du dataset Pima Indians Diabetes...
Dataset charge : 768 lignes x 9 colonnes
Distribution : {0: 500, 1: 268}
Train : 614 | Test : 154
Entrainement : RF_baseline
/home/zeco/Desktop/TPs/mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values.

In [ ]:
# we should run the ui from the same folder in which we launched the experiments. 
! mlflow ui

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/07 20:56:08 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/07 20:56:08 INFO:     Started parent process [196065]
2026/05/07 20:56:12 INFO:     Started server process [196069]
2026/05/07 20:56:12 INFO:     Waiting for application startup.
2026/05/07 20:56:12 INFO:     Application startup complete.
2026/05/07 20:56:13 INFO:     Started server process [196068]
2026/05/07 20:56:13 INFO:     Waiting for application startup.
2026/05/07 20:56:13 INFO:     Application startup complete.
2026/05/07 20:56:13 INFO:     Started server process [196071]
2026/05/07 20:56:13 INFO:     Waiting for application startup.
2026/05/07 20:56:13 INFO:     Applicatio

question 5a:  
in the context of illness detection, accuracy doesn't mean much, especially that we found the target class to be the minority, so a model that predicts only the healthy class would give a high accuracy but is useless.   
Recall represents how much can our model detect the target minority class (minimize false negatives), which is crusial in the medical context where classifying a positive case as negative costs lives. 


question 5b:   
class_weight hyperparam is used to assign importance coeffs to different classes, the way it works in sklearn is that it multiplies the loss contribution of each sample from the class by the class weight.  
when set to 'balanced', each class weight is measured using = n_total_samples/ (n_classes * n_class_samples), which makes the weight of each class inversly proportional to its presence in the dataset, to give more importance to minority class and restore some balance. 


question 6:  
we can detect overfitting by explicitely logging the gap between train and test metrics, then comparing it across runs, or we can plot train and test metrics in the Parallel Coordinate Plot and observe the gap